#### 1

In [32]:
import os
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from utils.timefeatures import time_features

import warnings
warnings.filterwarnings('ignore')

class Dataset_ETT_hour(Dataset):
    def __init__(self, root_path='./Dataset/', flag='train', size=None, 
                 features='S', data_path='ETTh1.csv', 
                 target='OT', scale=True, timeenc=0, freq='h'):
        # size [seq_len, label_len, pred_len]
        # info
        if size == None:
            self.seq_len = 24*4*4
            self.label_len = 24*4
            self.pred_len = 24*4
        else:
            self.seq_len = size[0]
            self.label_len = size[1]
            self.pred_len = size[2]
        # init
        assert flag in ['train', 'test', 'val']
        type_map = {'train':0, 'val':1, 'test':2}
        self.set_type = type_map[flag]
        
        self.features = features
        self.target = target
        self.scale = scale
        self.timeenc = timeenc
        self.freq = freq
        
        self.root_path = root_path
        self.data_path = data_path
        self.__read_data__()

    def __read_data__(self):
        self.scaler = StandardScaler()
        df_raw = pd.read_csv(os.path.join(self.root_path,
                                          self.data_path))

        border1s = [0, 12 * 30 * 24 - self.seq_len, 12 * 30 * 24 + 4 * 30 * 24 - self.seq_len]
        border2s = [12 * 30 * 24, 12 * 30 * 24 + 4 * 30 * 24, 12 * 30 * 24 + 8 * 30 * 24]
        border1 = border1s[self.set_type]
        border2 = border2s[self.set_type]
        
        if self.features=='M' or self.features=='MS':
            cols_data = df_raw.columns[1:]
            df_data = df_raw[cols_data]
        elif self.features=='S':
            df_data = df_raw[[self.target]]

        if self.scale:
            train_data = df_data[border1s[0]:border2s[0]]
            self.scaler.fit(train_data.values)
            data = self.scaler.transform(df_data.values)
        else:
            data = df_data.values
            
        df_stamp = df_raw[['date']][border1:border2]
        df_stamp['date'] = pd.to_datetime(df_stamp.date)
        if self.timeenc == 0:
            df_stamp['month'] = df_stamp.date.apply(lambda row: row.month, 1)
            df_stamp['day'] = df_stamp.date.apply(lambda row: row.day, 1)
            df_stamp['weekday'] = df_stamp.date.apply(lambda row: row.weekday(), 1)
            df_stamp['hour'] = df_stamp.date.apply(lambda row: row.hour, 1)
            data_stamp = df_stamp.drop(['date'], axis=1).values
        elif self.timeenc == 1:
            data_stamp = time_features(pd.to_datetime(df_stamp['date'].values), freq=self.freq)
            data_stamp = data_stamp.transpose(1, 0)

        self.data_x = data[border1:border2]
        self.data_y = data[border1:border2]
        self.data_stamp = data_stamp
    
    def __getitem__(self, index):
        s_begin = index
        s_end = s_begin + self.seq_len
        r_begin = s_end - self.label_len 
        r_end = r_begin + self.label_len + self.pred_len

        seq_x = self.data_x[s_begin:s_end]
        seq_y = self.data_y[r_begin:r_end]
        seq_x_mark = self.data_stamp[s_begin:s_end]
        seq_y_mark = self.data_stamp[r_begin:r_end]

        return seq_x, seq_y, seq_x_mark, seq_y_mark
    
    def __len__(self):
        return len(self.data_x) - self.seq_len - self.pred_len + 1

    def inverse_transform(self, data):
        return self.scaler.inverse_transform(data)
    
class Dataset_Custom(Dataset):
    def __init__(self, root_path='./Dataset/', flag='train', size=None,
                 features='S', data_path='ETTh1.csv',
                 target='OT', scale=True, timeenc=0, freq='h'):
        # size [seq_len, label_len, pred_len]
        # info
        if size == None:
            self.seq_len = 24 * 4 * 4
            self.label_len = 24 * 4
            self.pred_len = 24 * 4
        else:
            self.seq_len = size[0]
            self.label_len = size[1]
            self.pred_len = size[2]
        # init
        assert flag in ['train', 'test', 'val']
        type_map = {'train': 0, 'val': 1, 'test': 2}
        self.set_type = type_map[flag]

        self.features = features
        self.target = target
        self.scale = scale
        self.timeenc = timeenc
        self.freq = freq

        self.root_path = root_path
        self.data_path = data_path
        self.__read_data__()

    def __read_data__(self):
        self.scaler = StandardScaler()
        df_raw = pd.read_csv(os.path.join(self.root_path,
                                          self.data_path))

        '''
        df_raw.columns: ['date', ...(other features), target feature]
        '''
        cols = list(df_raw.columns)
        cols.remove(self.target)
        cols.remove('date')
        df_raw = df_raw[['date'] + cols + [self.target]]
        # print(cols)
        num_train = int(len(df_raw) * 0.7)
        num_test = int(len(df_raw) * 0.2)
        num_vali = len(df_raw) - num_train - num_test
        border1s = [0, num_train - self.seq_len, len(df_raw) - num_test - self.seq_len]
        border2s = [num_train, num_train + num_vali, len(df_raw)]
        border1 = border1s[self.set_type]
        border2 = border2s[self.set_type]

        if self.features == 'M' or self.features == 'MS':
            cols_data = df_raw.columns[1:]
            df_data = df_raw[cols_data]
        elif self.features == 'S':
            df_data = df_raw[[self.target]]

        if self.scale:
            train_data = df_data[border1s[0]:border2s[0]]
            self.scaler.fit(train_data.values)
            # print(self.scaler.mean_)
            # exit()
            data = self.scaler.transform(df_data.values)
        else:
            data = df_data.values

        df_stamp = df_raw[['date']][border1:border2]
        df_stamp['date'] = pd.to_datetime(df_stamp.date)
        if self.timeenc == 0:
            df_stamp['month'] = df_stamp.date.apply(lambda row: row.month, 1)
            df_stamp['day'] = df_stamp.date.apply(lambda row: row.day, 1)
            df_stamp['weekday'] = df_stamp.date.apply(lambda row: row.weekday(), 1)
            df_stamp['hour'] = df_stamp.date.apply(lambda row: row.hour, 1)
            data_stamp = df_stamp.drop(['date'], axis=1).values
        elif self.timeenc == 1:
            data_stamp = time_features(pd.to_datetime(df_stamp['date'].values), freq=self.freq)
            data_stamp = data_stamp.transpose(1, 0)

        self.data_x = data[border1:border2]
        self.data_y = data[border1:border2]
        self.data_stamp = data_stamp

    def __getitem__(self, index):
        s_begin = index
        s_end = s_begin + self.seq_len
        r_begin = s_end - self.label_len
        r_end = r_begin + self.label_len + self.pred_len

        seq_x = self.data_x[s_begin:s_end]
        seq_y = self.data_y[r_begin:r_end]
        seq_x_mark = self.data_stamp[s_begin:s_end]
        seq_y_mark = self.data_stamp[r_begin:r_end]

        return seq_x, seq_y, seq_x_mark, seq_y_mark

    def __len__(self):
        return len(self.data_x) - self.seq_len - self.pred_len + 1

    def inverse_transform(self, data):
        return self.scaler.inverse_transform(data)


In [33]:
def data_provider(config):
    if config.dataset == "ETTh1.csv" or config.dataset == "ETTh2.csv":
        train_dataset = Dataset_ETT_hour(
            flag='train',
            features='M', # M: multivariate, S: univariate
            size=[config.seq_len, config.label_len, config.pred_len],
            data_path=config.dataset
        )

        test_dataset = Dataset_ETT_hour(
            flag='test',
            features='M', # M: multivariate, S: univariate
            size=[config.seq_len, config.label_len, config.pred_len],
            data_path=config.dataset
        )
        
        vali_dataset = Dataset_ETT_hour(
            flag='val',
            features='M', # M: multivariate, S: univariate
            size=[config.seq_len, config.label_len, config.pred_len],
            data_path=config.dataset
        )
    elif config.dataset == "electricity.csv":
        train_dataset = Dataset_Custom(
            flag='train',
            features='M', # M: multivariate, S: univariate
            size=[config.seq_len, config.label_len, config.pred_len],
            data_path=config.dataset
        )

        test_dataset = Dataset_Custom(
            flag='test',
            features='M', # M: multivariate, S: univariate
            size=[config.seq_len, config.label_len, config.pred_len],
            data_path=config.dataset
        )
        
        vali_dataset = Dataset_Custom(
            flag='val',
            features='M', # M: multivariate, S: univariate
            size=[config.seq_len, config.label_len, config.pred_len],
            data_path=config.dataset
        )

    train_loader = DataLoader(
        dataset=train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=10,
        drop_last=True,
    )

    test_loader = DataLoader(
        dataset=test_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=10,
        drop_last=False
    )

    vali_loader = DataLoader(
        dataset=vali_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=10,
        drop_last=True,
        
    )

    return train_dataset, train_loader, test_dataset, test_loader, vali_dataset, vali_loader

In [34]:
from models import Linear, DMSLinear, LSTM, SparseTSF, SparseTSF_Transfrom, SparseTSF_Transfrom2
from utils.metric import metric
from utils.tools import EarlyStopping, adjust_learning_rate

import time
import matplotlib.pyplot as plt
import torch
from torch.optim import lr_scheduler 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
path = os.path.join('./checkpoints/', 'ETT_hour_linear.pth')
if not os.path.exists(path):
    os.makedirs(path)

In [ ]:
def train(config, train_loader):
    model_dict = {
        'Linear': Linear,
        'DMSLinear': DMSLinear,
        'LSTM': LSTM,
        'SparseTSF': SparseTSF,
        'SparseTSF_Transfrom': SparseTSF_Transfrom,
        'SparseTSF_Transfrom2' : SparseTSF_Transfrom2,
    }
    model = model_dict[config.model].Model(config).float().to(device)

    train_steps = len(train_loader)
    early_stopping = EarlyStopping(patience=config.patience, verbose=True)

    if config.criterion == 'MSE':
        criterion = torch.nn.MSELoss()
    elif config.criterion == 'MAE':
        criterion = torch.nn.L1Loss()
    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
    scheduler = lr_scheduler.OneCycleLR(optimizer = optimizer,
                                        steps_per_epoch = train_steps,
                                        pct_start = 0.3,
                                        epochs = config.epochs,
                                        max_lr = config.learning_rate,)

    for epoch in range(config.epochs):
        iter_count = 0
        train_loss = []
        
        model.train()
        epoch_time = time.time()
        for i, (batch_x, batch_y, batch_x_mark, batch_y_mark) in enumerate(train_loader):
            iter_count += 1
            optimizer.zero_grad()
            
            batch_x = batch_x.float().to(device)
            batch_y = batch_y.float().to(device)
            batch_x_mark = batch_x_mark.float().to(device)
            batch_y_mark = batch_y_mark.float().to(device)

            outputs = model(batch_x)
            outputs = outputs[:, -config.pred_len:, :]
            batch_y = batch_y[:, -config.pred_len:, :].to(device)

            loss = criterion(outputs, batch_y)
            train_loss.append(loss.item())
            
            loss.backward()
            optimizer.step()
            
        print("Epoch: {} cost time: {}".format(epoch + 1, time.time() - epoch_time))
        train_loss = np.average(train_loss)

        adjust_learning_rate(optimizer, scheduler, epoch + 1, config)
        print('=========================================')
    
    return loss

#### 2

In [ ]:
import optuna
import random
import torch
from types import SimpleNamespace

seed = 2023
random.seed(seed)
torch.manual_seed(seed)
np.random.seed(seed)

##################################
# 하이퍼파라미터 설정
def objective(pred_len, seg_len_x, seg_len_y, batch_size, learning_rate):
    config = SimpleNamespace(
        model='SparseTSF_Transfrom2',   # e.g., 'Linear', 'DMSLinear', 'SparseTSF_Transfrom', 'SparseTSF'
        dataset="ETTh2.csv", # ETTh1, ETTh2, traffic, electricity, traffic
        channels=7, # ETT = 7 / electricity = 321 / traffic = 862
        period_len=24,
        seq_len=720,
        pred_len=pred_len,
        seg_len_x=seg_len_x,  
        seg_len_y=seg_len_y, 
        batch_size=batch_size, # ETT = 256 / electricity & traffic = 128
        learning_rate=learning_rate,   
        label_len=0,                          
        lradj='type3', 
        epochs=30,               
        patience=5,
        criterion='MSE',  # Options: 'MSE', 'MAE'
    )

    # 실행 코드
    _, train_loader, _, _, _, _ = data_provider(config)
    loss = train(config, train_loader) 
    
    return loss